# 🧠 Machine Learning Fundamentals & scikit-learn Reference

## 1. What Is Machine Learning?

Machine Learning (ML) is a branch of artificial intelligence that builds algorithms which **learn patterns from data** to make predictions, group observations, or support decisions, with little manual rule-writing.

### Traditional data analysis vs machine learning

| Dimension | Traditional analysis | Machine learning |
| --- | --- | --- |
| **Starting point** | Hypothesis-driven | Data-driven |
| **Pattern search** | Manual queries and statistical tests | Automated pattern discovery |
| **Validation** | $p$-values and confidence intervals | Performance on unseen data (train/test split, cross-validation) |
| **Flexibility** | Bound by distribution assumptions (e.g. normality) | Adapts to non-linear, high-dimensional data |

### Data analysis objectives

| Objective | Question | Techniques |
| --- | --- | --- |
| Descriptive | What happened? | Summaries, visualisation |
| Exploratory | What patterns exist? | Pattern discovery, clustering |
| Confirmatory | Is the effect real? | Statistical testing |
| Predictive | What will happen? | Regression, classification, forecasting |
| Prescriptive | What should we do? | Decision optimisation |

## 2. Types of Machine Learning Problems

```text
Machine Learning
 │
 ├── Supervised learning (needs labelled targets y)
 │    ├── Regression: continuous target, y ∈ ℝ (e.g. sales forecast, house price)
 │    └── Classification: discrete target, y ∈ {0, 1, …, K−1} (e.g. churn, fraud, diagnosis)
 │
 └── Unsupervised learning (finds structure in the features X alone)
      ├── Clustering: groups similar observations (e.g. customer segments)
      ├── Association rules: finds items that co-occur (e.g. market baskets)
      └── Dimensionality reduction: compresses correlated features into fewer factors (e.g. PCA)
```

| Problem type | Learning mode | Goal | Typical algorithms | Example use case |
| --- | --- | --- | --- | --- |
| **Regression** | Supervised | Predict a continuous number | Linear / Ridge / Lasso, polynomial, SVR | Quarterly revenue, energy usage |
| **Classification** | Supervised | Predict a class label | Logistic regression, decision trees, SVM, neural networks | Churn, spam filtering, diagnosis |
| **Clustering** | Unsupervised | Group observations by similarity | K-Means, DBSCAN, hierarchical (Ward) | Customer segmentation, topic modelling |
| **Association rules** | Unsupervised | Find frequently co-occurring items | Apriori, FP-Growth | Market basket analysis, recommendations |
| **Dimensionality reduction** | Unsupervised | Reduce the number of features while keeping structure | PCA, t-SNE, UMAP | Student skill profiling |

### Worked example: student skill profiling
- **Problem:** 30+ course grades per student make strengths hard to see.
- **Approach:** group correlated grades (manually, by clustering the courses, or with PCA) into a few factors such as **technical**, **business** and **collaborative** skills.
- **Result:** each student is described by 3 interpretable scores instead of 30 grades.

## 3. Logistic Regression vs Decision Trees

Both are standard classification algorithms, but they build decision boundaries in fundamentally different ways.

| Criterion | Logistic regression | Decision tree |
| --- | --- | --- |
| **Model type** | Linear model (parametric); estimates class probabilities with the sigmoid function | Rule-based tree (non-parametric) |
| **Decision boundary** | **Linear** (a single hyperplane) | **Non-linear**, axis-aligned splits |
| **Interpretability** | Coefficients / odds ratios | Flowchart of `if–else` rules |
| **Feature scaling** | **Recommended** (affects regularisation and solver convergence) | **Not needed** (splits depend only on value order) |
| **Categorical features** | Need encoding (e.g. one-hot) | Handled well after encoding; splits are simple thresholds |
| **Missing values** | Need imputation | Supported natively in scikit-learn ≥ 1.3 |
| **Overfitting risk** | Lower; controlled with $L_1$ / $L_2$ penalties | Higher; limit with `max_depth`, `min_samples_leaf` |
| **Noise sensitivity** | Fairly robust | Sensitive — small data changes can change the tree |
| **Speed** | Fast, even on high-dimensional data | Fast for shallow trees; deep trees grow costly |

### When to use which
- **Logistic regression:** the classes are roughly linearly separable, you need calibrated probabilities, or you want coefficient-based explanations.
- **Decision tree:** relationships are non-linear, you want human-readable rules, or features are mixed and only lightly pre-processed.

```python
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report

lr = LogisticRegression(random_state=42).fit(X_train, y_train)
dt = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train, y_train)

print("=== Logistic Regression ===")
print(classification_report(y_test, lr.predict(X_test)))
print("=== Decision Tree ===")
print(classification_report(y_test, dt.predict(X_test)))
```

Useful metrics: accuracy, precision, recall, F1-score, ROC-AUC and the confusion matrix (see `05.03` and `08-some-notes`).

## 4. scikit-learn API: `fit`, `transform`, `predict`

scikit-learn separates **transformers** (pre-process data) from **estimators/predictors** (models).

| Class type | Example | `.fit()` | `.transform()` | `.fit_transform()` | `.predict()` | `.fit_predict()` |
| --- | --- | --- | --- | --- | --- | --- |
| Transformer | `StandardScaler` | ✅ | ✅ | ✅ | ❌ | ❌ |
| Transformer | `PolynomialFeatures` | ✅ | ✅ | ✅ | ❌ | ❌ |
| Supervised model | `LinearRegression` | ✅ | ❌ | ❌ | ✅ | ❌ |
| Supervised model | `DecisionTreeClassifier` | ✅ | ❌ | ❌ | ✅ | ❌ |
| Supervised model | `RandomForestClassifier` | ✅ | ❌ | ❌ | ✅ | ❌ |
| Unsupervised model | `KMeans` | ✅ | ✅ *(distances to centroids)* | ✅ | ✅ | ✅ |

### What each method does
- **`.fit(X, [y])`** — learns parameters from the data.
  - Scalers: compute each feature's mean ($\mu$) and standard deviation ($\sigma$).
  - Models: compute weights, split thresholds, centroids, etc.
- **`.transform(X)`** — changes $X$ using the parameters learned in `.fit()` (transformers, plus a few models such as `KMeans` and `PCA`).
- **`.fit_transform(X)`** — `.fit(X)` then `.transform(X)` in one call; use it on **training data only**.
- **`.predict(X)`** — returns predicted labels or values from a fitted model.
- **`.fit_predict(X)`** — fits and returns labels for the same data in one call; typical for clustering (`KMeans`, `DBSCAN`). `DBSCAN` has no separate `.predict()` for new points.

🎓 **Why does `PolynomialFeatures` have `.transform()` but no `.predict()`?** It is a transformer: it creates new features ($x^2$, $x_1x_2$, …) but learns no relationship to a target. A model such as `LinearRegression` then learns the weights and predicts.

> **Data leakage rule:** call `.fit_transform()` on the **training** split and only `.transform()` on the **validation/test** split, so test data is scaled with statistics learned from training data. A `Pipeline` does this for you.